In [104]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

In [52]:
pumpkins = pd.read_csv('../data/US-pumpkins.csv')
pumpkins.shape

(1757, 26)

to be able to use neccesary columns I will inspect and delete columns that have mostly non values

In [53]:
pumpkins.isnull().sum()

City Name             0
Type               1712
Package               0
Variety               5
Sub Variety        1461
Grade              1757
Date                  0
Low Price             0
High Price            0
Mostly Low          103
Mostly High         103
Origin                3
Origin District    1626
Item Size           279
Color               616
Environment        1757
Unit of Sale       1595
Quality            1757
Condition          1757
Appearance         1757
Storage            1757
Crop               1757
Repack                0
Trans Mode         1757
Unnamed: 24        1757
Unnamed: 25        1654
dtype: int64

City name package variety date low price high price 
mostly low after that point all of them need some fillment
mostly high
Origin 
item size 
color
repack this column almost identical among examples so I am not gonna be using it 

In [54]:
pumpkins['Variety'].value_counts()

Variety
HOWDEN TYPE                 542
PIE TYPE                    468
MINIATURE                   310
FAIRYTALE                   132
CINDERELLA                   81
BIG MACK TYPE                74
MIXED HEIRLOOM VARIETIES     57
HOWDEN WHITE TYPE            49
KNUCKLE HEAD                 20
BLUE TYPE                    19
Name: count, dtype: int64

In [55]:
pumpkins['Repack'].value_counts(dropna=False)

Repack
N    1752
E       5
Name: count, dtype: int64

In [56]:
pumpkins['Package'].value_counts(dropna=False)
print(pumpkins['Package'].head(20))

0     24 inch bins
1     24 inch bins
2     24 inch bins
3     24 inch bins
4     24 inch bins
5     24 inch bins
6     36 inch bins
7     36 inch bins
8     36 inch bins
9     36 inch bins
10    36 inch bins
11    36 inch bins
12    36 inch bins
13    36 inch bins
14    36 inch bins
15    36 inch bins
16    36 inch bins
17    36 inch bins
18    36 inch bins
19    36 inch bins
Name: Package, dtype: str


In [57]:
pumpkins['Origin'].value_counts(dropna=False)
pumpkins.dropna(subset=['Origin'], inplace=True)
# we only have 3 null values so it is safe to drop them
# we have 613 empty color values I will check if there is a way 
# to repair them, if not I will drop them as well

In [58]:
# changing package variable 
# I delete each since its not a fixed amount and I will convert the rest to bushels
pumpkins.drop(pumpkins[pumpkins['Package'] == 'each'].index, inplace=True)
conversion_dict = {
    'bushel cartons': 1.0,
    'bushel baskets': 1.0,
    '1/2 bushel cartons': 0.5,
    '1 1/9 bushel cartons': 1.11,
    '1 1/9 bushel crates': 1.11,
    '40 lb cartons': 1.0,
    '50 lb cartons': 1.25,
    '50 lb sacks': 1.25,
    '35 lb cartons': 0.875,
    '22 lb cartons': 0.55,
    '20 lb cartons': 0.5,
    '36 inch bins': 25.0,
    '24 inch bins': 18.0,
    'bins': 25.0,
    }
#  these values are based of a bushel being 25 pounds

pumpkins['Package'] = pumpkins['Package'].map(conversion_dict)

In [59]:
neccessary_columns = ['Item Size','City Name','Date', 'Variety', 'Origin', 'Color', 'Package', 'Low Price', 'High Price']
pumpkins = pumpkins[neccessary_columns]
print(pumpkins.head())

  Item Size  City Name     Date      Variety    Origin   Color  Package  \
0       lge  BALTIMORE  4/29/17          NaN  MARYLAND     NaN     18.0   
1       lge  BALTIMORE   5/6/17          NaN  MARYLAND     NaN     18.0   
2       med  BALTIMORE  9/24/16  HOWDEN TYPE  DELAWARE  ORANGE     18.0   
3       med  BALTIMORE  9/24/16  HOWDEN TYPE  VIRGINIA  ORANGE     18.0   
4       lge  BALTIMORE  11/5/16  HOWDEN TYPE  MARYLAND  ORANGE     18.0   

   Low Price  High Price  
0      270.0       280.0  
1      270.0       280.0  
2      160.0       160.0  
3      160.0       160.0  
4       90.0       100.0  


In [62]:
pumpkins['Date'] = pd.to_datetime(pumpkins['Date'], dayfirst=True)
print(pumpkins['Date'].head())

0   2017-04-29
1   2017-06-05
2   2016-09-24
3   2016-09-24
4   2016-05-11
Name: Date, dtype: datetime64[us]


In [63]:
pumpkins['Year'] = pumpkins['Date'].dt.year
pumpkins['Month'] = pumpkins['Date'].dt.month
pumpkins['Day'] = pumpkins['Date'].dt.day

In [64]:
pumpkins.drop(columns=['Date'], inplace=True)

In [65]:
pumpkins.head(5)

,Item Size,City Name,Variety,Origin,Color,Package,Low Price,High Price,Year,Month,Day
0,lge,BALTIMORE,NaN,MARYLAND,NaN,18.0,270.0,280.0,2017,4,29
1,lge,BALTIMORE,NaN,MARYLAND,NaN,18.0,270.0,280.0,2017,6,5
2,med,BALTIMORE,HOWDEN TYPE,DELAWARE,ORANGE,18.0,160.0,160.0,2016,9,24
3,med,BALTIMORE,HOWDEN TYPE,VIRGINIA,ORANGE,18.0,160.0,160.0,2016,9,24
4,lge,BALTIMORE,HOWDEN TYPE,MARYLAND,ORANGE,18.0,90.0,100.0,2016,5,11


In [66]:
pd.set_option('display.max_rows', None)
pumpkins.groupby(['Item Size', 'Origin','Color'])['Variety'].unique()
# I tried diffrent variations to specify null variety values but I could not find a pattern to fill them in. I will drop them since they are only 613 rows out of 100k rows.
pumpkins.dropna(subset=['Variety'], inplace=True)

In [67]:
pumpkins.dropna(inplace=True)

In [68]:
pumpkins.shape

(991, 11)

In [69]:
item_size_categories = [['sml', 'med', 'med-lge', 'lge', 'xlge', 'jbo', 'exjbo']]
ordinal_features = ['Item Size']
ordinal_encoder = OrdinalEncoder(categories=item_size_categories)

In [83]:
categorical_features = ['City Name', 'Variety', 'Origin']
categorical_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [84]:
label_features = ['Color']
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(pumpkins['Color'])

In [90]:
ct = ColumnTransformer([
    ('ordinal', ordinal_encoder, ordinal_features),
    ('categorical', categorical_encoder, categorical_features)
])
ct.set_output(transform='pandas')
encoded_features = ct.fit_transform(pumpkins)

In [92]:
encoded_features.shape

(991, 41)

In [93]:
x_train, x_test, y_train, y_test = train_test_split(encoded_features, encoded_labels, test_size=0.2, random_state=42)

In [96]:
model = LogisticRegression()

model.fit(x_train, y_train)

predictions = model.predict(x_test)

model.score(x_test, y_test)

0.8994974874371859

In [105]:
pipeline = make_pipeline(
    PolynomialFeatures(degree=4),
    StandardScaler(), 
    LogisticRegression()
)
pipeline.fit(x_train, y_train)

pipeline.score(x_test, y_test)

c:\Users\abdullah\Desktop\MlStart\ML-For-Beginners\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8944723618090452

even when I deleted unnecesary data it seems like before the lessons they really work for it 
only change that I made was using day month year instead of dayofthe year and it still makes a big difference in the model